# 像素展示.py
**電腦視覺與深度學習 › Day14 OpenCV 電腦視覺**

這一課的第一支程式,只做一件事:把一張照片讀進電腦、顯示出來。前面幾天我們處理的是文字和數字,今天開始處理影像。

> 🎧 有逐段旁白、逐句拆解的版本在 [pyl Academy](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/0/2)。
>
> **怎麼用**:由上往下,每格按左邊 ▶ 執行;或選單「執行階段 › 全部執行」。
>
> 📷 Colab 沒有攝影機、也開不了視窗:下面會**改用範例圖片**代替鏡頭,影像直接顯示在格子底下(只顯示前幾張)。

In [ ]:
#@title 0. 準備環境(下載課程範例檔、安裝套件)
!git clone -q --depth 1 https://github.com/friencdking0000/2026AI-STUST.git /content/2026AI-STUST 2>/dev/null; echo 課程檔案已就緒
%cd "/content/2026AI-STUST/Week1/DAY7"
import sys, os; sys.argv = sys.argv[:1]  # 讓 argparse / sys.argv 在 notebook 裡也正常
__file__ = os.path.join(os.getcwd(), '像素展示.py')  # notebook 沒有 __file__,程式靠它找同資料夾的檔案

In [ ]:
#@title 1. Colab 沒有攝影機和視窗:改用範例圖片、影像直接顯示在下面
import cv2, numpy as np, urllib.request, os
from google.colab.patches import cv2_imshow
def _pyl_sample():
    img = np.full((480, 640, 3), 255, np.uint8)
    cv2.circle(img, (130, 140), 70, (0, 0, 255), -1)
    cv2.circle(img, (520, 360), 55, (255, 0, 0), -1)
    cv2.rectangle(img, (260, 60), (400, 200), (0, 160, 0), -1)
    cv2.fillPoly(img, [np.array([[90, 420], [230, 420], [160, 300]])], (0, 200, 255))
    cv2.fillPoly(img, [np.array([[300, 300], [430, 280], [450, 420], [320, 440]])], (200, 0, 200))
    return img
class _PylCam:
    def __init__(self, *a, **k): self.n = 0; self.img = _pyl_sample()
    def isOpened(self): return True
    def read(self):
        self.n += 1
        return (True, self.img.copy()) if self.n <= 30 else (False, None)
    def set(self, *a): return True
    def get(self, *a): return 0
    def release(self): pass
_pyl_VC = cv2.VideoCapture
cv2.VideoCapture = lambda src=0, *a, **k: _PylCam() if isinstance(src, int) else _pyl_VC(src, *a, **k)
_pyl_shown = [0]
def _pyl_imshow(name, img):
    _pyl_shown[0] += 1
    if _pyl_shown[0] <= 6:
        print('▼', name); cv2_imshow(img)
cv2.imshow = _pyl_imshow
cv2.waitKey = lambda *a: ord('q') if _pyl_shown[0] >= 6 else -1
cv2.destroyAllWindows = lambda *a: None
cv2.namedWindow = lambda *a, **k: None
cv2.createTrackbar = lambda *a, **k: None
cv2.getTrackbarPos = lambda *a, **k: 0
cv2.setMouseCallback = lambda *a, **k: None
cv2.resizeWindow = cv2.moveWindow = cv2.setWindowProperty = cv2.setWindowTitle = lambda *a, **k: None
_pyl_imread = cv2.imread
def _pyl_imread2(path, *a, **k):
    img = _pyl_imread(path, *a, **k)
    if img is None and not os.path.exists(str(path)):
        print('(找不到', path, ':課程 repo 沒附這張圖,改用範例圖片)')
        img = _pyl_sample()
    return img
cv2.imread = _pyl_imread2
print('已改用範例圖片代替攝影機')

### 請出 OpenCV
第一行把 OpenCV 請進來,在 Python 裡它的名字叫 cv2。第二行的 os 是處理檔案路徑的小幫手,等一下組圖片路徑會用到。

In [ ]:
import cv2

### 用程式所在的資料夾組路徑
這三行先找出這支程式自己放在哪個資料夾,再把圖片檔名接上去。這樣整個資料夾搬到別台電腦也找得到圖。舊版課本是把講師電腦的完整路徑寫死,換電腦就會失敗,現在改成這種寫法了。

In [ ]:
import os
# 1. 讀取圖片
# 用「腳本自己所在的資料夾」為基準來組路徑（類似 ROS 的 cd <workspace> 再用相對路徑）
# 這樣把整個資料夾搬到別台電腦、別的位置，圖片一樣抓得到，不會寫死在某個絕對路徑
資料夾 = os.path.dirname(os.path.abspath(__file__))          # 這支 .py 檔所在的資料夾
圖片路徑 = os.path.join(資料夾, 'WIN_20260703_09_51_29_Pro.jpg')  # 圖片與腳本放同一層

### 讀圖並檢查有沒有讀到
imread 會把圖片變成一大堆數字,也就是一個矩陣。要特別記住:路徑錯了它不會報錯,只會默默回傳 None。所以下一行一定要先檢查 img 是不是 None,是的話就印出提示並結束。

In [ ]:
# 第二個參數 cv2.IMREAD_COLOR 代表讀取彩色圖片（預設值）
img = cv2.imread(圖片路徑)

### 顯示視窗、等你按鍵
imshow 開一個視窗把矩陣畫成圖。waitKey 括號裡的 0 代表一直等,直到你在視窗上按任何一個鍵。最後 destroyAllWindows 把 OpenCV 開的視窗全部關掉。

In [ ]:
# 2. 檢查圖片是否成功讀取（路徑寫錯或檔案損壞會讀到 None）
if img is None:
    print(f"錯誤：無法讀取圖片，請檢查檔案是否存在！\n嘗試讀取的路徑：{圖片路徑}")
    exit()
# 3. 顯示圖片
# 第一個參數是視窗名稱，第二個參數是要顯示的影像矩陣
cv2.imshow('Image Viewer', img)
# 4. 等待使用者按下鍵盤任意鍵
# 參數 0 代表無限期等待，直到按下鍵盤為止
cv2.waitKey(0)

# 5. 關閉所有 OpenCV 建立的視窗
cv2.destroyAllWindows()

## 小結
你學會了讀圖、檢查 None、顯示和關閉視窗,這是所有影像程式的骨架。下一支「視覺」會把一張圖換成攝影機的連續畫面。

👉 回到 [pyl Academy 這堂課](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/0/2) 看逐句拆解、做練習。